# Ferrari Luce AudienceKit Walkthrough

This notebook shows the intended library-first workflow: load a weighted audience frame, sample respondents, load a structured study, run the panel, and summarize results.

In [ ]:
from pathlib import Path

import audiencekit as ak
from audiencekit.report import likert_summary, plot_likert, write_html_report

ROOT = Path.cwd()
STUDY_PATH = ROOT / "examples" / "ferrari_luce" / "study.json"
OUT_DIR = ROOT / "examples" / "ferrari_luce" / "results"

In [ ]:
pool = ak.load_panel()
respondents = ak.sample_panel(pool, n=25, segment="broad", seed=42)
respondents[["id", "age", "sex", "region", "income16", "segment"]].head()

In [ ]:
study = ak.Study.from_json(STUDY_PATH)
study.question_ids

The next cell makes live model calls. Set `OPENAI_API_KEY` first, or inject your own backend object that implements `get_completion(prompt, image=None, **kwargs)`.

In [ ]:
panel = ak.SyntheticPanel(respondents, backend_type="openai", temperature=0.7)
results = panel.run_survey(study)
OUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT_DIR / "broad_default_responses.csv", index=False)
likert_summary(results, study)

In [ ]:
quotes = results["first_reaction"].dropna().head(5).tolist()
write_html_report(
    OUT_DIR / "report.html",
    study,
    results,
    figures=[plot_likert(results, study)],
    quotes=quotes,
)